# Integrated Intelligent Systems: Tutorial 3 - Body Motion Problem (Scientific Object) (3 Pts)

This notebook introduces a formulation body motion problem, central to designing and implementing Cognition-Enabled Robot Agents, as a description logic system.

## Part 1: Robot-World Manipulation System as Scientific Object (E1–E6) (1.5 Pt)

**Title:** Formalizing the Robot Agent: From Architecture to Description Logic Reasoners

![Alternative Text](RWMS.png)

### Scenario Background
An autonomous robotic agent (the PR2 in the EASE apartment) is deployed to execute a manipulation task ("Serve Eva a drink"). To validate this agent scientifically, we must formalize the system as a **Robot-World Manipulation System (RWMS)**, separate the controlling subsystem from the controlled body-world system, define trajectories as trees, and assess runs against an external problem standard using a description logic reasoner.

---

### Multi-Question Student Task Sheet (0.25 Pt x 6)

1. **E1 · Architectural Cut & Subsystem Separation**
   - In the graphical representation of the RWMS, a strict architectural "cut" is placed between the manipulation executive and the body-world system ($[B \bowtie W]$). Explain the nature of this cut: why is the manipulation executive classified as a `dul:Agent` (non-physical program) while the body-world system is classified under physical objects, and how do they interact exclusively across the boundary?
  
   - The RWMS is divided into two subsystems by a strict **cut between the executive and the body**.

- The **manipulation executive** reads observations, decides and commits to action tokens. It is a program with no mass, pose or physical contact, so it is classified as a non-physical `dul:Agent`.
- The **body–world system (BWS)**, written [B ⋈ W], contains the PR2, the apartment and the fridge. It is made of physical objects that evolve under physics, so it is classified under physical objects.

The two sides interact **only across the cut**:
- **Action tokens** go from the executive to the BWS.
- **Observations** go from the BWS back to the executive.

The axioms enforce this: `ManipulationExecutive ⊑ ∃controls.BodyWorldSystem ⊓ ∀actsOn.BodyWorldSystem`, so the executive cannot act on the carton directly (that is inconsistent). The manipulation problem P stands outside the system and reaches it only through the trajectories it classifies.

2. **E2 · Forward-Branching Time & World Choices**
   - During an operation on a fridge door or container seal, the same action token (e.g., a position-controlled pull) can lead to completely divergent physical outcomes depending on world choices (e.g., seal releases vs. seal holds tight). Draw or describe the structure of the resulting courses, and explain why the possible courses form a tree that branches *only forward* from the initial state $x_0$.
  
   - The same position-controlled pull on the fridge door can lead to two courses: the **seal releases** and the door swings open, or the **seal holds tight** and the handle is torn off.

Structure of the courses (tree from the initial state x₀):

| Executive chooses (one branch must work) | World chooses (every branch must work) | Leaf class |
|---|---|---|
| position-controlled pull | seal releases | SOL (drink at Eva's place) |
| position-controlled pull | seal holds tight | PROH (handle torn off, P4) |
| force-limited pull (up to F_max) | seal releases | SOL |
| force-limited pull (up to F_max) | seal holds tight | REPD (door jolts, carton tilts 20°, settles) |
| decline, report | n/a | NON (authorised decline) |

The possible courses form a tree that **branches only forward** because:
- Every state has exactly one history leading back to x₀, and branches never merge.
- Alternatives appear only at the next step, where the executive picks a token and the world picks its response (for example a stiffer seal).
- Physical events such as tearing or spilling cannot be undone, so the future is open and the past is fixed.


3. **E3 · Trajectory Classification & Priority Hierarchies**
   - The problem standard $P$ sorts every physical trajectory into one of four classes: **SOL** (Solution), **REPD** (Repaired), **NON** (Nonachievement), and **PROH** (Prohibited). Consider two runs that both successfully place the drink within Eva's reach, but where Run A tears off the container handle (`P4` violation) while Run B maintains licensed contact. Classify both runs and explain why the classifier grades the *entire course* rather than just the final end-state.
   - - **Run A (handle torn off, P4 violated): PROH.**
- **Run B (licensed contact maintained): SOL**, assuming all achievements and maintenance conditions hold. It would be REPD only if it had departed from a maintain condition within an admitted band and returned.

The classes are applied in priority order: **PROH › NON › REPD › SOL**. A violated prevention outranks goal achievement, so Run A is PROH even though the drink is within Eva's reach.

The classifier grades the **entire course** because the conditions are over the whole run (`¬OccursIn(Damage(c), H)` over the horizon H), not over its end state. Two runs can have the same end state, yet one may have torn the handle off on the way. An end-state check could not tell them apart.


4. **E4 · Description Logic Partonomy & DUL Constraints**
   - When formalizing the RWMS in description logic under DOLCE Ultra Lite (DUL), we define an RWMS as having *exactly one* executive and *exactly one* body-world system (`RWMS = (=1 hasComponent.ManipExecutive) ⊓ (=1 hasComponent.BodyWorldSystem)`). Using the transitivity of `dul:isPartOf`, explain how a reasoner (such as HermiT) automatically entails that the PR2's head camera is a component of the RWMS without explicitly hardcoding every part-whole link.
  
   - Only direct part-whole links are asserted:
- `Sensor ⊑ ∃dul:isPartOf.RobotBody` (the head camera is part of the body)
- `BodyWorldSystem ⊑ (=1 dul:hasComponent.RobotBody)` (the body is a component of the BWS)
- `RWMS ≡ (=1 dul:hasComponent.ManipulationExecutive) ⊓ (=1 dul:hasComponent.BodyWorldSystem)` (the BWS is a component of the RWMS)

In DUL, `hasComponent ⊑ hasProperPart ⊑ hasPart`, and `dul:isPartOf` is **transitive**. A reasoner such as HermiT therefore chains the links: camera is part of body, body is part of BWS, BWS is part of RWMS, so the camera is part of the RWMS. The answer is entailed, so no part-whole pair needs hard-coding.


5. **E5 · Ontological Inconsistency & Entailment Limits**
   - An inconsistency in description logic serves as a strict analytical tool. Formulate a competency question that results in an ontological inconsistency when attempting to classify the *manipulation problem itself* as a physical component inside the RWMS. Furthermore, explain why numerical physical laws (such as Coulomb friction $2 \cdot \mu \cdot F \ge m \cdot (g + a)$) must be handled outside description logic by the semantic digital twin's causal model.
     
**Competency question:** "Can the manipulation problem be a part of the RWMS?"

To test it, add the statement and ask whether the ontology is still consistent. It is not. Because `RWMS ⊑ ∀dul:hasComponent.(ManipulationExecutive ⊔ dul:PhysicalObject)`, and the problem is neither, together with DUL disjointness, the reasoner reports an **inconsistent value restriction**. The problem is related to the system only through `RWMS ⊑ ∃assessedAgainst.ManipulationProblem`: it is the standard the system is judged by, not a component.

**Why numerical laws stay outside description logic:**
- DL has **no arithmetic**. The Coulomb condition 2·μ·F ≥ m·(g + a) is a calculation over real-valued force, friction and acceleration, not a class membership.
- OWL 2 forbids counting (`=1`) along transitive roles such as `dul:hasPart`, because reasoning would become undecidable.

So the law belongs to the **semantic digital twin's causal model**, while the ontology supplies the vocabulary.



6. **E6 · Strength vs. Truth in Agent Validation**
   - The design thesis dictates that an agent decides within its believed causality, which can be tested for logical *strength* via competency queries and for *truth* via physical experiments. Compare how an axiom set that lacks a friction law ($\text{K}_0$) performs on competency queries versus an axiomatization containing explicit dynamical laws ($\text{K}_1/\text{K}_2$), and state what kind of test is required to validate the actual physical truth of those beliefs.
  
   - An agent decides how to accomplish an intended action **within the causality it believes**. Competency queries test the **strength** of an axiomatization, and physical experiments test its **truth**.

| Axiom set | Result on the wet-carton questions |
|---|---|
| K0: ontology + dynamics, no friction law | No answer to any question (too weak) |
| K1: K0 + A1 (Coulomb) + values of μ, m, F | Answers the holds / slips / minimum-μ questions correctly; no answer on test-then-commit |
| K2: K1 + A2 (a slip shifts the pinch) + A6 (what the body sees) | Answers all four questions |
| Kx: K0 + "a pinch above 15 N always holds" | Answers everything, but wrongly (strong but untrue) |

A set without a friction law is silent on the competency queries, while explicit dynamical laws make it entail answers. Strength is not truth, as Kx shows. To validate the truth of the beliefs we need a **physical experiment**: carry a real wet carton on the real robot and compare the measured slip with the predicted slip edge.


## Part 2: Robot Motion Problem and Description Logics (1.5 Pt)

**Title:** Formulating and Querying the Robot Motion Problem with Knowledge Bases & Description Logics

![Alternative Text](PR2.png)

### Scenario Background
Using the interactive Jupyter notebook and RViz simulation environment (`bmp.rviz`, `setup_demo()`), the PR2 mobile manipulator interacts with the EASE apartment model (including the fridge and various kitchen drawers). Instead of hardcoding trajectories, the robotic executive queries a knowledge base (represented via Description Logics / Prolog reasoning) to link task requests to state changes, causes (motions), and capabilities.

[Go to the programming Lab](https://binder.intel4coro.de/v2/gh/maltehue/giskard-examples/bmp_vrb?urlpath=/lab/tree/notebooks/bodyMotionProblem.ipynb)
---

### Multi-Question Student Task Sheet (0.25 Pt x 6)

1. **E7 · Knowledge Base Grounding & Task Requests**
   - Execute the initial query `taskRequest(Request, StateChange).`Examine the resulting solution bindings (e.g., `{'Request': 'open_fridge', 'StateChange': 'open(fridge)'}` and various drawer operations). Explain how mapping abstract task requests to discrete state changes forms the top-level specification layer for the manipulation executive.

2. **E8 · Extending Queries for Causal Motions & Parameters**
   - Extend the query to incorporate motion causes: `taskRequest(Request, StateChange), causes(Motion, StateChange, MotionParam).` Explain how the reasoner bridges the gap between a desired symbolic state change and continuous motion parameters (such as `envJointGoal`, target joint angles, and handle identifiers).

3. **E9 · Integrating Robot Capabilities (`canPerform`)**
   - Finalize the query by adding the robot capability constraint: `taskRequest(Request, StateChange), causes(Motion, StateChange, MotionParam), canPerform(Robot, Motion, MotionParam).`Describe what role the `Robot` variable (`'pr2'`) plays in filtering feasible trajectories through the reasoner.

4. **E10 · Specializing Request Scope (`open_fridge` vs. `open_anything`)**
   - Modify the query request parameter from general categories like `open_anything` to specifically target `open_fridge`. Explain how refining the scope in the knowledge base reduces the solution search tree and prevents unauthorized or unintended state changes in the kitchen environment.

5. **E11 · Visualization and Simulation Linkage in RViz**
   - With the background shell command executing `rviz -d .../bmp.rviz` and displaying the PR2 interacting with the fridge handle, explain how the motion parameters derived from the reasoner's output (`[0.8, b'fridge_door_joint', b'fridge_door_handle']`) are executed as physical joint trajectories in the simulation.

6. **E12 · Ontological Limits & Missing Dynamics**
   - While the knowledge base successfully infers *which* motion achieves a state change, explain why a pure Description Logic or symbolic Prolog reasoner cannot verify whether a specific grasp force will cause a wet carton to slip, and identify which subsystem must handle those physical calculations.
  
   - E12 · Ontological Limits & Missing Dynamics
A pure Description Logic or Prolog reasoner can identify which motion achieves a desired state change, but cannot verify whether a grasping force will cause a wet carton to slip.
- Lack of numerical computation: DL cannot evaluate physical equations involving friction (μ), force (F), mass (m), and acceleration (a), such as the Coulomb friction condition.
- Missing physical dynamics: It cannot independently predict real-world behaviour such as slipping due to reduced friction on a wet carton.
These calculations must be handled by the semantic digital twin's causal physics model or a physics simulator, supported by the robot's force or tactile sensors.

Conclusion: The reasoner determines what to do, while the physics model and robot controller determine whether it is physically feasible and safe.

# Submission guidelines

- Deadline: 24.11.2026, 4 PM
- Submission as Git repo commits pushed within deadlines (If not yet, create a repo with the name **iis-assignment-win26** for the whole semester and invite me as collaborator)
- Put the textual answers in notebook and all related media (images, rules, etc) should then be put into a folder called **Assignment3**. 